# Лабораторная работа № 4. Метод опорных векторов (SVM) с ядрами

**Курс:** Классическое машинное обучение, 4 курс прикладной математики

## Цель работы

Изучить метод опорных векторов, научиться решать двойственную задачу и использовать ядра для нелинейной классификации.

**Используемые инструменты:** `scipy.optimize`, `sklearn.svm`, `matplotlib`, опционально `cvxopt`.

### Регламент сдачи

Работа сдаётся в виде этого же ноутбука, дополненного вашим кодом. Обязательно:

1. Читаемый код с комментариями.
2. Визуализации (графики, таблицы).
3. **Текстовый вывод после каждого задания** — не только код, но и объяснение результата.
4. Финальный вывод по работе.

**Критерии оценки:** корректность реализации — 30 %, качество визуализаций и анализа — 20 %,
обоснованность выводов — 20 %, сравнение с эталонными реализациями — 15 %,
оригинальность и дополнительная работа — 15 %.

> Ячейки, помеченные `# TODO`, нужно заполнить самостоятельно.
> Ячейки с готовым кодом можно просто выполнить — они подготавливают данные и графики.

## Подготовка окружения

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})
sns.set_palette("viridis")

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

from sklearn.datasets import make_blobs, make_moons, make_circles, load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

X_lin, y_lin = make_blobs(n_samples=80, centers=2, cluster_std=0.9, random_state=RANDOM_STATE)
y_lin = np.where(y_lin == 0, -1, 1)          # для SVM метки удобнее в виде -1 / +1

plt.scatter(X_lin[:, 0], X_lin[:, 1], c=y_lin, cmap="coolwarm", edgecolor="k", s=45)
plt.title("Линейно разделимая выборка")
plt.show()

## Задание 1. Двойственная задача линейного SVM

Двойственная задача:

$$\max_{\alpha}\ \sum_{i=1}^{n}\alpha_i - \frac{1}{2}\sum_{i,j}\alpha_i \alpha_j y_i y_j \langle x_i, x_j\rangle
\quad \text{при} \quad \sum_i \alpha_i y_i = 0,\ \ 0 \le \alpha_i \le C.$$

Восстановление ответа: $w = \sum_i \alpha_i y_i x_i$, а **опорные векторы** — объекты с $\alpha_i > 0$.

Решать будем через `scipy.optimize.minimize` (метод SLSQP поддерживает и равенства, и границы).

In [ ]:
from scipy.optimize import minimize

def svm_dual_fit(X, y, C=1.0):
    """Решение двойственной задачи линейного SVM.

    Возвращает (alpha, w, b).
    """
    n = len(y)
    K = X @ X.T                       # матрица Грама для линейного ядра
    P = np.outer(y, y) * K

    # TODO:
    #  1. objective(a)  = 0.5 * a @ P @ a - a.sum()      (минимизируем -L)
    #  2. jac(a)        = P @ a - 1
    #  3. ограничение   {"type": "eq", "fun": lambda a: a @ y}
    #  4. границы       [(0, C)] * n
    #  5. res = minimize(objective, np.zeros(n), jac=jac, bounds=..., constraints=..., method="SLSQP")
    #  6. w = ((alpha * y) @ X); b — по любому объекту с 0 < alpha < C
    raise NotImplementedError

In [ ]:
# TODO: найдите опорные векторы (alpha > 1e-5) и визуализируйте:
#   - точки выборки, раскрашенные по классам;
#   - разделяющую прямую w @ x + b = 0;
#   - границы зазора w @ x + b = ±1 (пунктиром);
#   - опорные векторы — обведёнными крупными кружками.
# Сравните w и b с SVC(kernel="linear", C=...).

## Задание 2. Ядра на нелинейных данных

Ядро позволяет работать в спрямляющем пространстве, не вычисляя само отображение:

$$K_{\text{lin}}(x, z) = \langle x, z\rangle, \qquad
  K_{\text{poly}}(x, z) = (\gamma\langle x, z\rangle + r)^d, \qquad
  K_{\text{RBF}}(x, z) = \exp\bigl(-\gamma\|x - z\|^2\bigr).$$

In [ ]:
X_nl, y_nl = make_moons(n_samples=300, noise=0.25, random_state=RANDOM_STATE)

def plot_decision_boundary(model, X, y, ax, title):
    """Готовая функция для отрисовки разделяющей поверхности."""
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    zz = model.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, zz, levels=20, cmap="coolwarm", alpha=0.7)
    ax.contour(xx, yy, zz, levels=[0], colors="k", linewidths=1.5)
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolor="k", s=25)
    ax.set_title(title)
    ax.grid(False)

In [ ]:
# TODO: обучите SVC с kernel="linear" и kernel="rbf" на make_moons,
#       нарисуйте обе границы через plot_decision_boundary в одной строке subplots.
#       Повторите на make_circles. Какое ядро справляется и почему?

## Задание 3. Подбор $C$ и $\gamma$ через GridSearchCV

* $C$ — штраф за ошибки: большое $C$ — узкий зазор и риск переобучения, малое — широкий зазор.
* $\gamma$ — «радиус влияния» объекта в RBF: большое $\gamma$ — модель запоминает каждую точку.

In [ ]:
cancer = load_breast_cancer()
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    cancer.data, cancer.target, test_size=0.3, stratify=cancer.target, random_state=RANDOM_STATE)

# TODO: соберите Pipeline(StandardScaler -> SVC(kernel="rbf")) и запустите GridSearchCV
#       по сетке C in [0.1, 1, 10, 100], gamma in [1e-3, 1e-2, 1e-1, 1], scoring="roc_auc", cv=5

## Задание 4. Тепловая карта качества

Постройте heatmap: строки — $C$, столбцы — $\gamma$, значения — AUC на кросс-валидации.
Отметьте оптимальную ячейку.

In [ ]:
# TODO: возьмите grid.cv_results_["mean_test_score"], сделайте reshape по сетке
#       и постройте sns.heatmap(..., annot=True, fmt=".3f")

**Вывод:** *как ведёт себя качество при слишком большом γ? Что это говорит о переобучении?*

## Дополнительное задание. Полиномиальное ядро вручную

Реализуйте функцию `poly_kernel(X, Z, degree=3, gamma=1.0, coef0=1.0)` и подставьте её
в `SVC(kernel=<ваша функция>)` (sklearn принимает callable). Сравните с RBF по качеству
и по времени обучения.

In [ ]:
def poly_kernel(X, Z, degree=3, gamma=1.0, coef0=1.0):
    # TODO: (gamma * X @ Z.T + coef0) ** degree
    raise NotImplementedError


# TODO: SVC(kernel=poly_kernel) — обучите и сравните с SVC(kernel="rbf")

## Финальный вывод

*Напишите здесь связный вывод по работе (5–10 предложений):*

- какие методы вы применили и почему;
- какие результаты получили в числах;
- где реализация «с нуля» разошлась с эталоном из `sklearn` и в чём причина;
- что бы вы улучшили, будь у вас больше времени.